In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system.
2. Open a terminal and run:

```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

If you want to test that the local server is running, you can also run:

```bash
curl http://localhost:11434
```

If you get a connection refused error while prompting Ollama, restart the server with:

```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I couldn’t find any FAQ entry about **Olama** specifically.

If you meant **running the course locally**, the FAQ says you can do that instead of using Codespaces, as long as you’re comfortable setting up:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

It also says to **document your setup** and keep your environment **reproducible**.

If you meant something else by “Olama,” let me know the exact tool/name and I can check the FAQ context again.


In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Maybe — but I need a bit more context.\n\nAre you asking about:\n- a specific class/course at a school or university,\n- an online course,\n- or an internal training course?\n\nIn general, whether you can join depends on things like:\n- whether enrollment is still open,\n- prerequisites,\n- capacity,\n- and the instructor or organizer’s approval.\n\nIf you tell me the course name and where it’s offered, I can help you figure out the next step or draft a message asking to join.'

In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [9]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [10]:
len(response.output)

1

In [11]:
call = response.output[0]

In [12]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join if I just discovered the course? enrollment late join eligibility join course discovered course"}', call_id='call_oelDpDSHHGm8iTc8g27qdia7', name='search', type='function_call', id='fc_0960266731aa4779006a33970c7ee881a09ad36d6a3de8d166', namespace=None, status='completed')

In [13]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join if I just discovered the course? enrollment late join eligibility join course discovered course'}

In [14]:
call.name

'search'

In [15]:
results = search(**args)

In [16]:
result_json = json.dumps(results, indent=2)

In [17]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [18]:
messages.append(call)

In [19]:
messages.append(function_call_output)

In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join if I just discovered the course? enrollment late join eligibility join course discovered course"}', call_id='call_oelDpDSHHGm8iTc8g27qdia7', name='search', type='function_call', id='fc_0960266731aa4779006a33970c7ee881a09ad36d6a3de8d166', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_oelDpDSHHGm8iTc8g27qdia7',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the cou

In [21]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [22]:
print(response.output_text)

Yes — you can still join and start learning.

If you want a certificate, though, you’ll need to submit your project while the course is still accepting submissions.


In [23]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(779, 37)

In [24]:
def calculate_openai_cost(
    input_tokens: int,
    output_tokens: int,
    input_price_per_million: float = 0.25,
    output_price_per_million: float = 2.00,
) -> float:
    """
    Calculate OpenAI API cost in USD.

    Args:
        input_tokens: Number of input tokens.
        output_tokens: Number of output tokens.
        input_price_per_million: USD per 1M input tokens.
        output_price_per_million: USD per 1M output tokens.

    Returns:
        Total cost in USD.
    """
    input_cost = (input_tokens / 1_000_000) * input_price_per_million
    output_cost = (output_tokens / 1_000_000) * output_price_per_million
    return input_cost + output_cost

In [25]:
cost = calculate_openai_cost(779, 37)
print(f"${cost:.8f}")

$0.00026875
